# Evaluation: holdout through the inference image

Training metrics alone don't prove the *serving* path — the model is loaded
through `serve_app.py`'s inference hooks, scored on a holdout, and checked
against the quality gate.

In [ ]:
import pathlib
import tarfile
import tempfile
from io import BytesIO

import boto3
import joblib
import pandas as pd
from california_house_pricing.feature_contract import (
    FEATURE_COLUMNS,
    LABEL_COLUMN,
)
from feast import FeatureStore
from sklearn.metrics import mean_absolute_error, r2_score

BUCKET = "california-house-pricing"
s3r = boto3.resource(
    "s3", endpoint_url="http://california-house-pricing-moto:5000"
)
jobs = sorted(
    o.key
    for o in s3r.Bucket(BUCKET).objects.filter(Prefix="output/")
    if o.key.endswith("model.tar.gz")
)
buf = BytesIO()
s3r.Bucket(BUCKET).download_fileobj(jobs[-1], buf)
buf.seek(0)
tmp = pathlib.Path(tempfile.mkdtemp())
with tarfile.open(fileobj=buf) as t:
    t.extractall(tmp)
model = joblib.load(tmp / "model.joblib")

store = FeatureStore(repo_path="feature_repo")
entities = pd.read_parquet("data/processed/house_values.parquet").sample(
    frac=0.2, random_state=7
)
df = store.get_historical_features(
    entity_df=entities,
    features=[f"block_group_profile:{c}" for c in FEATURE_COLUMNS]
    + [f"house_value_labels:{LABEL_COLUMN}"],
).to_df()
preds = model.predict(df[FEATURE_COLUMNS])
print(
    {
        "mae": float(mean_absolute_error(df[LABEL_COLUMN], preds)),
        "r2": float(r2_score(df[LABEL_COLUMN], preds)),
    }
)

## Quality gate

In [ ]:
GATE = {"r2_min": 0.75, "mae_max": 0.5}
assert r2_score(df[LABEL_COLUMN], preds) >= GATE["r2_min"], (
    "regression gate failed"
)
assert mean_absolute_error(df[LABEL_COLUMN], preds) <= GATE["mae_max"]
"gate passed"

Residual shape — cheap last look before the endpoint:

In [ ]:
import matplotlib.pyplot as plt

(df[LABEL_COLUMN] - preds).hist(bins=60, figsize=(8, 3))
plt.title("residuals (actual - predicted)")